真实的哈特曼传感器程序

In [ ]:
from pathlib import Path
from pprint import pprint

import numpy as np

# SI units: m, rad, px, unless the key name says otherwise.
PROJECT_DIR = Path.cwd()

params = {
    "source": {
        "wavelength_m": 561e-9,
    },
    "camera": {
        "resolution_px": (1440, 1080),
        "pixel_pitch_m": 3.45e-6,
    },
    "microlens_array": {
        "model": "MLAS10-F15-P300-AB",
        "pitch_m": 300e-6,
        "focal_length_m": 14.6e-3,
    },
    "relay": {
        "dm_to_sh_magnification": 1.0,
    },
    "dm": {
        "sdk_dir": str(PROJECT_DIR / "python3.9_bmc"),
        "serial_number": "17DW013#137",
        "flat_map_file": str(PROJECT_DIR / "17DW013#137_FLAT_MAP_COMMANDS.txt"),
        "active_actuator_array": (12, 12),
        "available_actuators": 140,
        "active_aperture_m": 4.4e-3,
        "stroke_m": 3500e-9,
        "command_range": (0.0, 1.0),
        "flat_command": 0.5,
    },
}

camera_size_m = np.array(params["camera"]["resolution_px"]) * params["camera"]["pixel_pitch_m"]
dm_image_size_m = params["dm"]["active_aperture_m"] * params["relay"]["dm_to_sh_magnification"]

derived_params = {
    "camera_size_mm": tuple(camera_size_m * 1e3),
    "lenslet_pitch_px": params["microlens_array"]["pitch_m"] / params["camera"]["pixel_pitch_m"],
    "dm_image_size_mm": dm_image_size_m * 1e3,
    "dm_image_size_px": dm_image_size_m / params["camera"]["pixel_pitch_m"],
    "dm_lenslets_across": dm_image_size_m / params["microlens_array"]["pitch_m"],
}

pprint(params)
print("\nDerived:")
pprint(derived_params)


加载dm可变镜的控制库

In [ ]:
import os
import sys


def load_bmc_sdk(sdk_dir):
    sdk_dir = Path(sdk_dir)
    if not sdk_dir.exists():
        raise FileNotFoundError(f"BMC SDK directory not found: {sdk_dir}")

    if hasattr(os, "add_dll_directory"):
        os.add_dll_directory(str(sdk_dir))
    if str(sdk_dir) not in sys.path:
        sys.path.insert(0, str(sdk_dir))

    import bmc

    return bmc


def load_flat_commands(flat_map_file, expected_count, command_range):
    flat_map_file = Path(flat_map_file)
    if not flat_map_file.exists():
        raise FileNotFoundError(f"Flat map file not found: {flat_map_file}")

    commands = np.loadtxt(flat_map_file, dtype=np.float64).reshape(-1)
    if commands.size != expected_count:
        raise ValueError(f"Expected {expected_count} flat commands, got {commands.size}.")
    if not np.all(np.isfinite(commands)):
        raise ValueError("Flat commands contain non-finite values.")

    lo, hi = command_range
    if np.any((commands < lo) | (commands > hi)):
        raise ValueError(f"Flat commands must stay in [{lo}, {hi}].")
    return commands


bmc = load_bmc_sdk(params["dm"]["sdk_dir"])
flat_commands = load_flat_commands(
    params["dm"]["flat_map_file"],
    expected_count=params["dm"]["available_actuators"],
    command_range=params["dm"]["command_range"],
)

print(f"BMC SDK version: {bmc.BmcDm.version_string()}")
print(f"Flat map length: {flat_commands.size}")
print(f"Flat map command range: {flat_commands.min():.6f} to {flat_commands.max():.6f}")
print(f"Flat map mean command: {flat_commands.mean():.6f}")


将dm控制函数进行包装

In [ ]:
class BmcDmController:
    def __init__(self, bmc_module, dm_params, flat_commands):
        self.bmc = bmc_module
        self.dm_params = dm_params
        self.flat_commands = np.asarray(flat_commands, dtype=np.float64)
        self.dm = None

    def _check(self, code, action):
        if int(code) != int(self.bmc.NO_ERR):
            message = ""
            if self.dm is not None:
                message = self.dm.error_string(int(code))
            raise RuntimeError(f"{action} failed: code {code}. {message}")

    def open(self):
        serial_number = self.dm_params["serial_number"]
        if not serial_number:
            raise ValueError("Set params['dm']['serial_number'] before opening the DM.")

        self.dm = self.bmc.BmcDm()
        self._check(self.dm.open_dm(serial_number), "open_dm")

        actuator_count = int(self.dm.num_actuators())
        if actuator_count != self.flat_commands.size:
            self.close()
            raise RuntimeError(
                f"DM reports {actuator_count} actuators, but flat map has {self.flat_commands.size}."
            )

        print(f"Opened DM {serial_number}: {actuator_count} actuators")
        return self

    def send(self, commands, label="commands"):
        if self.dm is None:
            raise RuntimeError("Open the DM before sending commands.")

        commands = np.asarray(commands, dtype=np.float64).reshape(-1)
        if commands.size != self.flat_commands.size:
            raise ValueError(f"Expected {self.flat_commands.size} commands, got {commands.size}.")
        if not np.all(np.isfinite(commands)):
            raise ValueError(f"{label} contain non-finite values.")

        lo, hi = self.dm_params["command_range"]
        if np.any((commands < lo) | (commands > hi)):
            raise ValueError(f"{label} must stay in [{lo}, {hi}].")

        self._check(self.dm.send_data(self.bmc.DoubleVector(commands.tolist())), f"send {label}")
        return commands

    def send_flat(self):
        sent = self.send(self.flat_commands, label="flat map")
        actual = np.asarray(list(self.dm.get_actuator_data()), dtype=np.float64)
        print(f"Flat map sent. Stored-command max error: {np.max(np.abs(actual - sent)):.3e}")
        return sent

    def close(self):
        if self.dm is not None:
            code = self.dm.close_dm()
            self._check(code, "close_dm")
            self.dm = None
            print("DM closed. Note: BMC close_dm sets output to zero, not to the flat map.")


运行下面这个单元会打开真实 DM，并发送厂家 flat map，产生纯平平面。

In [ ]:
dm_ctrl = BmcDmController(bmc, params["dm"], flat_commands).open()
dm_ctrl.send_flat()
